# 🎬 AI Video & Anime Khmer Dubbing Studio - All-in-One Colab
### 👑 ដំណើរការទាំង Web Studio Tool + AI GPU Server (VoxCPM2) លើ Google Colab តែមួយ!

> **⚠️ សំខាន់បំផុតមុនចុច Run:**
> ចូលទៅកាន់ **Runtime ➔ Change runtime type ➔ ជ្រើសរើស T4 GPU ➔ រួចចុច Save**

In [ ]:
# 1. ពិនិត្យមើលកម្លាំង GPU (ត្រូវប្រាកដថាចេញ Tesla T4 16GB VRAM)
!nvidia-smi

In [ ]:
# 2. ដំឡើង Cloudflared, FFmpeg, និង Packages ទាំងអស់
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Clone VoxCPM (AI GPU Model)
!git clone https://github.com/OpenBMB/VoxCPM.git /content/VoxCPM 2>/dev/null || (cd /content/VoxCPM && git pull)
!pip install -q fastapi uvicorn python-multipart soundfile numpy nest_asyncio
!pip install -q einops addict wetext modelscope funasr argbind torchcodec
!pip install -q -e /content/VoxCPM

# Clone Web Studio Tool
!git clone https://github.com/cm5722254-beep/-animeclone.git /content/animeclone 2>/dev/null || (cd /content/animeclone && git pull)
!pip install -q pydub edge-tts google-generativeai requests python-dotenv
!apt-get install -y ffmpeg > /dev/null 2>&1

print("✅ បរិស្ថាន និងកូដទាំងអស់ត្រូវបានរៀបចំរួចរាល់ ១០០%!")

In [ ]:
# 3. ចាប់ផ្ដើមដំណើរការទាំង Web Studio & VoxCPM2 GPU រួមគ្នាក្នុង Colab តែមួយ
import os, sys, time, re, subprocess, threading

sys.path.insert(0, "/content/VoxCPM/src")
sys.path.insert(0, "/content/VoxCPM")
sys.path.insert(0, "/content/animeclone")

# 1. ចាប់ផ្ដើម Cloudflare Tunnel សម្រាប់ Web Studio (Port 3000)
tunnel_cmd = "cloudflared tunnel --url http://localhost:3000"
proc = subprocess.Popen(tunnel_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
def read_tunnel():
    global public_url
    for line in iter(proc.stdout.readline, ""):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m and not public_url:
            public_url = m.group(0)
            print("\n" + "=" * 70)
            print("🎉 ផ្ទាំង WEB STUDIO ONLINE រួចរាល់ហើយ!")
            print(f"👉 ចុចបើក Link នេះដើម្បីប្រើប្រាស់ Web Tool ផ្ទាល់: {public_url}")
            print("=" * 70 + "\n")

threading.Thread(target=read_tunnel, daemon=True).start()

# 2. ដំណើរការ Web Studio Backend លើ Port 3000
%cd /content/animeclone
os.environ['PORT'] = '3000'
!python server.py